# 08 · WM-811K 웨이퍼맵 결함 CNN (3단계)

실제 fab **웨이퍼맵 811,457장**, 결함패턴 **9종** 분류(CNN). 자동 결함 진단 → 공정 근본원인 추적의 제품성 컴포넌트.

> 데이터: Kaggle `qingyi/wm811k-wafer-map` → `data/raw/wm811k/LSWMD.pkl`.
> torch 필요: `pip install -r requirements-cnn.txt`

## 0. 준비
```bash
pip install -r requirements-cnn.txt
kaggle datasets download -d qingyi/wm811k-wafer-map -p data/raw/wm811k
cd data/raw/wm811k && unzip wm811k-wafer-map.zip
```

In [ ]:
import sys; sys.path.append("..")
import numpy as np, matplotlib.pyplot as plt
from src.datasets.wm811k import load_wm811k, LABELS
# 빠른 학습 위해 클래스당 상한 (none 압도 완화)
X, y, LABELS = load_wm811k(size=64, labeled_only=True, max_per_class=3000)
print("X:", X.shape, "| classes:", len(LABELS))

## 1. 클래스 분포 (극불균형 — none 다수)

In [ ]:
import collections
cnt = collections.Counter(y)
for i, l in enumerate(LABELS): print(f"{l:10s}: {cnt.get(i,0)}")

## 2. 샘플 시각화

In [ ]:
fig, ax = plt.subplots(1, 9, figsize=(16, 2))
for i, l in enumerate(LABELS):
    idx = np.where(y == i)[0]
    ax[i].imshow(X[idx[0], 0], cmap="viridis"); ax[i].set_title(l, fontsize=8); ax[i].axis("off")
plt.tight_layout(); plt.show()

## 3. CNN 학습 (클래스 가중 손실, macro-F1)

In [ ]:
from src.cnn import train_model, predict
model, history = train_model(X, y, n_classes=len(LABELS), epochs=8, batch=128)

## 4. 평가 — macro-F1 + 혼동행렬

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import seaborn as sns
_, Xte, _, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
pred = predict(model, Xte)
print("macro-F1:", round(f1_score(yte, pred, average="macro"), 4))
print(classification_report(yte, pred, target_names=LABELS))
cm = confusion_matrix(yte, pred)
sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS, cmap="Blues")
plt.xlabel("pred"); plt.ylabel("true"); plt.title("WM-811K confusion"); plt.tight_layout(); plt.show()

> **포인트**: Accuracy는 none 다수라 부풀려짐 → **macro-F1**·혼동행렬로 소수 결함패턴 성능을 정직하게 본다.
> 결함패턴 자동 분류 = fab 수율팀의 근본원인 분석 자동화 = 제품 가치.